
# City of London crime forecasting: simple XGBoost vs mixed pipeline

This notebook compares two approaches on the same complete monthly panel:

1. **Simple XGBoost benchmark**: similar to the existing teammate notebook, using only month, LSOA code, and crime type.
2. **Mixed forecasting pipeline**: improved XGBoost with lag/rolling features + Poisson count model + rolling baseline ensemble.

Expected folder layout:

```text
project_folder/
├── data/
│   ├── 2024-01/2024-01-city-of-london-street.csv
│   ├── ...
│   └── 2024-12/2024-12-city-of-london-street.csv
└── test/
    ├── 2025-01/2025-01-city-of-london-street.csv
    ├── ...
    └── 2025-12/2025-12-city-of-london-street.csv
```

If you use a different folder structure, only edit the `TRAIN_DIR` and `TEST_DIR` variables below.


In [ ]:

# Core imports
from pathlib import Path
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor

plt.style.use("default")
pd.set_option("display.max_columns", 100)


In [ ]:

# -----------------------------
# 1. Settings
# -----------------------------
TRAIN_DIR = Path("data")     # 2024 folders
TEST_DIR = Path("test")      # 2025 folders
FORCE_NAME = "city-of-london"

TRAIN_YEAR = 2024
TEST_YEAR = 2025

# If your 2025 data only has some months, the loader will use only the files that exist.


In [ ]:

# -----------------------------
# 2. Load monthly CSV files
# -----------------------------
def load_year_from_folders(base_dir: Path, year: int, force_name: str = "city-of-london") -> pd.DataFrame:
    frames = []
    missing = []

    for month in range(1, 13):
        folder_name = f"{year}-{month:02d}"
        file_name = f"{folder_name}-{force_name}-street.csv"
        file_path = base_dir / folder_name / file_name

        if file_path.exists():
            temp = pd.read_csv(file_path)
            temp["source_file"] = str(file_path)
            frames.append(temp)
        else:
            missing.append(str(file_path))

    if missing:
        print(f"Missing {len(missing)} files for {year}.")
        for p in missing[:5]:
            print("  -", p)
        if len(missing) > 5:
            print("  ...")

    if not frames:
        raise FileNotFoundError(f"No CSV files found for {year} in {base_dir.resolve()}")

    out = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(frames)} files for {year}. Shape: {out.shape}")
    return out

train_raw = load_year_from_folders(TRAIN_DIR, TRAIN_YEAR, FORCE_NAME)
test_raw = load_year_from_folders(TEST_DIR, TEST_YEAR, FORCE_NAME)

raw_all = pd.concat([train_raw, test_raw], ignore_index=True)
raw_all.head()


In [ ]:

# -----------------------------
# 3. Aggregate to monthly counts
# -----------------------------
needed_cols = ["Month", "LSOA code", "LSOA name", "Crime type"]
missing_cols = [c for c in needed_cols if c not in raw_all.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Keep only rows with valid LSOA and crime type.
# Some police.uk rows can have missing LSOA fields.
clean_raw = raw_all.dropna(subset=["Month", "LSOA code", "Crime type"]).copy()

monthly_counts = (
    clean_raw
    .groupby(["Month", "LSOA code", "LSOA name", "Crime type"])
    .size()
    .reset_index(name="crime_count")
)

monthly_counts["date"] = pd.to_datetime(monthly_counts["Month"] + "-01")
monthly_counts["year"] = monthly_counts["date"].dt.year
monthly_counts["month_num"] = monthly_counts["date"].dt.month

print("Aggregated shape:", monthly_counts.shape)
display(monthly_counts.head())


In [ ]:

# -----------------------------
# 4. Build a complete monthly panel
# -----------------------------
# Important: this creates explicit zero rows, so the model learns both crime and no-crime cases.

all_months = pd.date_range(
    monthly_counts["date"].min(),
    monthly_counts["date"].max(),
    freq="MS"
)

lsoa_lookup = (
    monthly_counts[["LSOA code", "LSOA name"]]
    .drop_duplicates("LSOA code")
    .sort_values("LSOA code")
)

crime_types = sorted(monthly_counts["Crime type"].dropna().unique())

full_index = pd.MultiIndex.from_product(
    [all_months, lsoa_lookup["LSOA code"], crime_types],
    names=["date", "LSOA code", "Crime type"]
).to_frame(index=False)

panel = full_index.merge(lsoa_lookup, on="LSOA code", how="left")
panel = panel.merge(
    monthly_counts[["date", "LSOA code", "Crime type", "crime_count"]],
    on=["date", "LSOA code", "Crime type"],
    how="left"
)

panel["crime_count"] = panel["crime_count"].fillna(0).astype(int)
panel["Month"] = panel["date"].dt.strftime("%Y-%m")
panel["year"] = panel["date"].dt.year
panel["month_num"] = panel["date"].dt.month

print("Complete panel shape:", panel.shape)
print("Date range:", panel["Month"].min(), "to", panel["Month"].max())
print("LSOAs:", panel["LSOA code"].nunique(), "| Crime types:", panel["Crime type"].nunique())
display(panel.head())


In [ ]:

# -----------------------------
# 5. Add forecasting features
# -----------------------------
def add_forecasting_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["LSOA code", "Crime type", "date"]).copy()
    group_cols = ["LSOA code", "Crime type"]
    g = df.groupby(group_cols)["crime_count"]

    # Lag features: previous crime levels for the same LSOA and crime type
    for lag in [1, 2, 3, 6, 12]:
        df[f"lag_{lag}"] = g.shift(lag)

    # Rolling features, shifted by 1 to avoid looking at the current target month
    df["rolling_3_mean"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    df["rolling_6_mean"] = g.transform(lambda s: s.shift(1).rolling(6, min_periods=1).mean())
    df["rolling_3_sum"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())
    df["rolling_6_sum"] = g.transform(lambda s: s.shift(1).rolling(6, min_periods=1).sum())

    # City-wide and crime-type context from previous month
    city_month_total = df.groupby("date")["crime_count"].sum().rename("city_total")
    crime_type_month_total = df.groupby(["date", "Crime type"])["crime_count"].sum().rename("crime_type_total")

    df = df.merge(city_month_total, on="date", how="left")
    df = df.merge(crime_type_month_total, on=["date", "Crime type"], how="left")

    df = df.sort_values(["LSOA code", "Crime type", "date"])
    df["city_total_lag_1"] = df.groupby(["LSOA code", "Crime type"])["city_total"].shift(1)
    df["crime_type_total_lag_1"] = df.groupby(["LSOA code", "Crime type"])["crime_type_total"].shift(1)

    # Calendar features
    df["quarter"] = df["date"].dt.quarter
    df["is_winter"] = df["month_num"].isin([12, 1, 2]).astype(int)
    df["is_summer"] = df["month_num"].isin([6, 7, 8]).astype(int)

    # Fill missing early lags with 0
    feature_cols_to_fill = [
        c for c in df.columns
        if c.startswith("lag_") or c.startswith("rolling_") or c.endswith("lag_1")
    ]
    df[feature_cols_to_fill] = df[feature_cols_to_fill].fillna(0)

    return df

model_df = add_forecasting_features(panel)

print("Model dataframe shape:", model_df.shape)
display(model_df.head())


In [ ]:
# -----------------------------
# 6. Train/validation/test split
# -----------------------------
# Train on 2024 and test on 2025.
# We also keep the last 3 months of 2024 as a validation block.
# This lets us tune the mixed-pipeline weights without cheating on the 2025 test set.

VALIDATION_MONTHS = [10, 11, 12]

full_train_df = model_df[model_df["year"] == TRAIN_YEAR].copy()
test_df = model_df[model_df["year"] == TEST_YEAR].copy()

inner_train_df = full_train_df[~full_train_df["month_num"].isin(VALIDATION_MONTHS)].copy()
val_df = full_train_df[full_train_df["month_num"].isin(VALIDATION_MONTHS)].copy()

# Final models are still trained on all 2024 data.
train_df = full_train_df.copy()

print("Inner train rows:", len(inner_train_df), "| Validation rows:", len(val_df), "| Test rows:", len(test_df))
print("Inner train months:", inner_train_df["Month"].min(), "to", inner_train_df["Month"].max())
print("Validation months:", val_df["Month"].min(), "to", val_df["Month"].max())
print("Test months:", test_df["Month"].min(), "to", test_df["Month"].max())

# For categorical consistency in XGBoost
cat_categories = {}
for col in ["LSOA code", "Crime type"]:
    categories = sorted(model_df[col].astype(str).unique())
    cat_categories[col] = categories
    for frame in [inner_train_df, val_df, train_df, test_df]:
        frame[col] = pd.Categorical(frame[col].astype(str), categories=categories)

In [ ]:

# -----------------------------
# 7. Metrics helper
# -----------------------------
def clip_predictions(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom

def evaluate_model(name, y_true, y_pred):
    y_pred = clip_predictions(y_pred)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "WAPE": wape(np.asarray(y_true), y_pred),
        "Bias": np.mean(y_pred - np.asarray(y_true)),
        "Actual total": np.sum(y_true),
        "Predicted total": np.sum(y_pred),
    }


In [ ]:
# -----------------------------
# 8. Model A: simple XGBoost benchmark
# -----------------------------
# This is close to the teammate model: Month + LSOA + Crime type only.

simple_features = ["month_num", "LSOA code", "Crime type"]

def prepare_xgb_frame(df, features):
    out = df[features].copy()
    for col in ["LSOA code", "Crime type"]:
        if col in out.columns:
            out[col] = pd.Categorical(out[col].astype(str), categories=cat_categories[col])
    return out

def make_simple_xgb():
    return xgb.XGBRegressor(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.04,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=2,
        reg_alpha=0.05,
        reg_lambda=1.0,
        objective="reg:squarederror",
        enable_categorical=True,
        tree_method="hist",
        random_state=42,
    )

# Validation version: train on Jan-Sep 2024, validate on Oct-Dec 2024.
X_inner_simple = prepare_xgb_frame(inner_train_df, simple_features)
y_inner = inner_train_df["crime_count"].values
X_val_simple = prepare_xgb_frame(val_df, simple_features)
y_val = val_df["crime_count"].values

simple_xgb_val = make_simple_xgb()
simple_xgb_val.fit(X_inner_simple, y_inner)
pred_val_simple_xgb = clip_predictions(simple_xgb_val.predict(X_val_simple))

# Final benchmark version: train on all 2024, test on 2025.
X_train_simple = prepare_xgb_frame(train_df, simple_features)
y_train = train_df["crime_count"].values
X_test_simple = prepare_xgb_frame(test_df, simple_features)
y_test = test_df["crime_count"].values

simple_xgb = make_simple_xgb()
simple_xgb.fit(X_train_simple, y_train)
pred_simple_xgb = clip_predictions(simple_xgb.predict(X_test_simple))

print("Validation:", evaluate_model("Simple XGBoost", y_val, pred_val_simple_xgb))
print("Test:", evaluate_model("Simple XGBoost", y_test, pred_simple_xgb))

In [ ]:
# -----------------------------
# 9. Model B1: improved XGBoost with lag/rolling features
# -----------------------------
# This model keeps the XGBoost idea but gives it actual forecasting information:
# previous crime levels, rolling trends, and city/crime-type context.

mixed_features = [
    "month_num", "quarter", "is_winter", "is_summer",
    "LSOA code", "Crime type",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "rolling_3_mean", "rolling_6_mean", "rolling_3_sum", "rolling_6_sum",
    "city_total_lag_1", "crime_type_total_lag_1",
]

def make_improved_xgb():
    return xgb.XGBRegressor(
        n_estimators=600,
        max_depth=3,
        learning_rate=0.025,
        subsample=0.9,
        colsample_bytree=0.85,
        min_child_weight=3,
        reg_alpha=0.10,
        reg_lambda=1.50,
        objective="count:poisson",   # better suited to non-negative count data
        enable_categorical=True,
        tree_method="hist",
        random_state=42,
    )

# Validation version
X_inner_mixed_xgb = prepare_xgb_frame(inner_train_df, mixed_features)
X_val_mixed_xgb = prepare_xgb_frame(val_df, mixed_features)

improved_xgb_val = make_improved_xgb()
improved_xgb_val.fit(X_inner_mixed_xgb, y_inner)
pred_val_improved_xgb = clip_predictions(improved_xgb_val.predict(X_val_mixed_xgb))

# Final version
X_train_mixed_xgb = prepare_xgb_frame(train_df, mixed_features)
X_test_mixed_xgb = prepare_xgb_frame(test_df, mixed_features)

improved_xgb = make_improved_xgb()
improved_xgb.fit(X_train_mixed_xgb, y_train)
pred_improved_xgb = clip_predictions(improved_xgb.predict(X_test_mixed_xgb))

print("Validation:", evaluate_model("Improved XGBoost", y_val, pred_val_improved_xgb))
print("Test:", evaluate_model("Improved XGBoost", y_test, pred_improved_xgb))

In [ ]:
# -----------------------------
# 10. Model B2: Poisson count model
# -----------------------------
# This gives the mixed pipeline a statistical count-model component.
# It is more interpretable than XGBoost and is appropriate for count data.

categorical_features = ["LSOA code", "Crime type"]
numeric_features = [c for c in mixed_features if c not in categorical_features]

def make_poisson_pipeline(alpha=0.5):
    # Compatibility for different sklearn versions
    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

    poisson_preprocess = ColumnTransformer(
        transformers=[
            ("cat", onehot, categorical_features),
            ("num", StandardScaler(), numeric_features),
        ],
        remainder="drop",
    )

    return Pipeline(steps=[
        ("preprocess", poisson_preprocess),
        ("model", PoissonRegressor(alpha=alpha, max_iter=2000)),
    ])

def prepare_sklearn_frame(df, features):
    out = df[features].copy()
    for col in categorical_features:
        out[col] = out[col].astype(str)
    return out

# Validation version
X_inner_poisson = prepare_sklearn_frame(inner_train_df, mixed_features)
X_val_poisson = prepare_sklearn_frame(val_df, mixed_features)

poisson_model_val = make_poisson_pipeline(alpha=0.5)
poisson_model_val.fit(X_inner_poisson, y_inner)
pred_val_poisson = clip_predictions(poisson_model_val.predict(X_val_poisson))

# Final version
X_train_poisson = prepare_sklearn_frame(train_df, mixed_features)
X_test_poisson = prepare_sklearn_frame(test_df, mixed_features)

poisson_model = make_poisson_pipeline(alpha=0.5)
poisson_model.fit(X_train_poisson, y_train)
pred_poisson = clip_predictions(poisson_model.predict(X_test_poisson))

print("Validation:", evaluate_model("Poisson count model", y_val, pred_val_poisson))
print("Test:", evaluate_model("Poisson count model", y_test, pred_poisson))

In [ ]:
# -----------------------------
# 11. Model B3: rolling baseline
# -----------------------------
# This is a simple model: recent average crime count for the same LSOA and crime type.
# It stabilizes the ensemble, but we will tune its weight instead of hard-coding it.

def rolling_baseline_prediction(df):
    return clip_predictions(
        df["rolling_3_mean"].fillna(df["lag_1"]).fillna(0).values
    )

pred_val_rolling_baseline = rolling_baseline_prediction(val_df)
pred_rolling_baseline = rolling_baseline_prediction(test_df)

print("Validation:", evaluate_model("Rolling baseline", y_val, pred_val_rolling_baseline))
print("Test:", evaluate_model("Rolling baseline", y_test, pred_rolling_baseline))

In [ ]:
# -----------------------------
# 12. Build mixed ensemble
# -----------------------------
# Instead of guessing the weights, this searches for the best ensemble weights
# on the Oct-Dec 2024 validation period. Then it applies those weights to the 2025 test set.
# This is better than tuning weights directly on 2025.

# Old fixed weights kept for comparison only.
W_XGB_FIXED = 0.60
W_POISSON_FIXED = 0.25
W_BASELINE_FIXED = 0.15

pred_mixed_pipeline_fixed = clip_predictions(
    W_XGB_FIXED * pred_improved_xgb +
    W_POISSON_FIXED * pred_poisson +
    W_BASELINE_FIXED * pred_rolling_baseline
)

pred_val_mixed_fixed = clip_predictions(
    W_XGB_FIXED * pred_val_improved_xgb +
    W_POISSON_FIXED * pred_val_poisson +
    W_BASELINE_FIXED * pred_val_rolling_baseline
)

# Validation-based ensemble search.
# XGBoost is allowed to dominate because the row-level data is sparse and nonlinear.
weight_results = []
for w_xgb in np.arange(0.50, 0.96, 0.05):
    for w_poisson in np.arange(0.00, 0.41, 0.05):
        w_baseline = 1.0 - w_xgb - w_poisson
        if w_baseline < -1e-9:
            continue

        pred_val_candidate = clip_predictions(
            w_xgb * pred_val_improved_xgb +
            w_poisson * pred_val_poisson +
            w_baseline * pred_val_rolling_baseline
        )

        result = evaluate_model("candidate", y_val, pred_val_candidate)
        result.update({
            "w_xgb": round(float(w_xgb), 2),
            "w_poisson": round(float(w_poisson), 2),
            "w_baseline": round(float(w_baseline), 2),
        })
        weight_results.append(result)

weight_results = pd.DataFrame(weight_results)
weight_results = weight_results.sort_values(["WAPE", "MAE", "RMSE"]).reset_index(drop=True)

best_weights = weight_results.iloc[0]
W_XGB = float(best_weights["w_xgb"])
W_POISSON = float(best_weights["w_poisson"])
W_BASELINE = float(best_weights["w_baseline"])

print("Best validation weights:")
print({"Improved XGBoost": W_XGB, "Poisson": W_POISSON, "Rolling baseline": W_BASELINE})
print("\nTop 10 validation weight combinations:")
display(weight_results.head(10))

# Optional level calibration learned on validation only.
# This corrects systematic over/under-prediction in total volume while avoiding extreme scaling.
pred_val_tuned_uncalibrated = clip_predictions(
    W_XGB * pred_val_improved_xgb +
    W_POISSON * pred_val_poisson +
    W_BASELINE * pred_val_rolling_baseline
)

raw_calibration_factor = np.sum(y_val) / max(np.sum(pred_val_tuned_uncalibrated), 1e-9)
CALIBRATION_FACTOR = float(np.clip(raw_calibration_factor, 0.70, 1.30))
print("Validation calibration factor:", CALIBRATION_FACTOR)

pred_mixed_pipeline_uncalibrated = clip_predictions(
    W_XGB * pred_improved_xgb +
    W_POISSON * pred_poisson +
    W_BASELINE * pred_rolling_baseline
)

pred_mixed_pipeline = clip_predictions(pred_mixed_pipeline_uncalibrated * CALIBRATION_FACTOR)

metrics = pd.DataFrame([
    evaluate_model("Simple XGBoost", y_test, pred_simple_xgb),
    evaluate_model("Improved XGBoost", y_test, pred_improved_xgb),
    evaluate_model("Poisson count model", y_test, pred_poisson),
    evaluate_model("Rolling baseline", y_test, pred_rolling_baseline),
    evaluate_model("Mixed pipeline fixed weights", y_test, pred_mixed_pipeline_fixed),
    evaluate_model("Mixed pipeline tuned", y_test, pred_mixed_pipeline),
])

# Make WAPE easier to read as percentage
metrics_display = metrics.copy()
metrics_display["WAPE"] = metrics_display["WAPE"] * 100

metrics_display.sort_values("MAE")

In [ ]:
# -----------------------------
# 13. Create one comparison results dataframe
# -----------------------------
results_df = test_df[["Month", "date", "month_num", "LSOA code", "LSOA name", "Crime type", "crime_count"]].copy()
results_df = results_df.rename(columns={"crime_count": "Actual"})

results_df["Simple XGBoost"] = pred_simple_xgb
results_df["Improved XGBoost"] = pred_improved_xgb
results_df["Poisson"] = pred_poisson
results_df["Rolling baseline"] = pred_rolling_baseline
results_df["Mixed pipeline fixed weights"] = pred_mixed_pipeline_fixed
results_df["Mixed pipeline"] = pred_mixed_pipeline

results_df["Simple XGBoost Error"] = results_df["Simple XGBoost"] - results_df["Actual"]
results_df["Mixed pipeline Error"] = results_df["Mixed pipeline"] - results_df["Actual"]

# Rounded predictions for dashboard-style tables.
# Keep continuous predictions for evaluation; rounded values are only for display.
results_df["Simple XGBoost rounded"] = np.round(results_df["Simple XGBoost"]).astype(int)
results_df["Mixed pipeline rounded"] = np.round(results_df["Mixed pipeline"]).astype(int)

results_df.head(10)

In [ ]:
# -----------------------------
# 14. Visualization 1: metric comparison
# -----------------------------
plot_metrics = metrics_display.set_index("model").loc[
    ["Simple XGBoost", "Mixed pipeline fixed weights", "Mixed pipeline tuned"],
    ["MAE", "RMSE", "WAPE"]
]

ax = plot_metrics.plot(kind="bar", figsize=(11, 6), rot=0)
ax.set_title("Model comparison: simple XGBoost vs mixed pipeline")
ax.set_ylabel("Metric value; WAPE shown as %")
ax.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

plot_metrics

In [ ]:
# -----------------------------
# 15. Visualization 2: monthly totals
# -----------------------------
monthly_summary = (
    results_df
    .groupby("month_num")[["Actual", "Simple XGBoost", "Mixed pipeline fixed weights", "Mixed pipeline"]]
    .sum()
    .reset_index()
)

month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly_summary["month_num"], monthly_summary["Actual"], marker="o", linewidth=2.5, label="Actual")
ax.plot(monthly_summary["month_num"], monthly_summary["Simple XGBoost"], marker="s", linestyle="--", linewidth=2.2, label="Simple XGBoost")
ax.plot(monthly_summary["month_num"], monthly_summary["Mixed pipeline fixed weights"], marker="^", linestyle=":", linewidth=2.0, label="Mixed pipeline fixed")
ax.plot(monthly_summary["month_num"], monthly_summary["Mixed pipeline"], marker="D", linestyle="--", linewidth=2.2, label="Mixed pipeline tuned")

ax.set_title("Monthly total crime: actual vs model predictions")
ax.set_xlabel("Month")
ax.set_ylabel("Total crime occurrences")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.legend()
ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

monthly_summary

In [ ]:
# -----------------------------
# 16. Visualization 3: totals by crime type
# -----------------------------
crime_summary = (
    results_df
    .groupby("Crime type")[["Actual", "Simple XGBoost", "Mixed pipeline"]]
    .sum()
    .sort_values("Actual", ascending=True)
)

ax = crime_summary.plot(kind="barh", figsize=(12, 8))
ax.set_title("Total 2025 crime by type: actual vs predictions")
ax.set_xlabel("Total occurrences")
ax.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

crime_summary

In [ ]:
# -----------------------------
# 17. Visualization 4: error by crime type
# -----------------------------
error_by_crime = results_df.groupby("Crime type").apply(
    lambda d: pd.Series({
        "Simple XGBoost MAE": mean_absolute_error(d["Actual"], d["Simple XGBoost"]),
        "Mixed pipeline MAE": mean_absolute_error(d["Actual"], d["Mixed pipeline"]),
        "Actual total": d["Actual"].sum(),
    })
).sort_values("Actual total", ascending=True)

ax = error_by_crime[["Simple XGBoost MAE", "Mixed pipeline MAE"]].plot(kind="barh", figsize=(12, 8))
ax.set_title("Prediction error by crime type")
ax.set_xlabel("Mean Absolute Error")
ax.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

error_by_crime

In [ ]:
# -----------------------------
# 18. Visualization 5: top LSOAs by actual demand
# -----------------------------
top_lsoas = results_df.groupby("LSOA code")["Actual"].sum().nlargest(15).index

lsoa_summary = (
    results_df[results_df["LSOA code"].isin(top_lsoas)]
    .groupby("LSOA code")[["Actual", "Simple XGBoost", "Mixed pipeline"]]
    .sum()
    .sort_values("Actual", ascending=True)
)

ax = lsoa_summary.plot(kind="barh", figsize=(12, 8))
ax.set_title("Top LSOAs: actual demand vs predictions")
ax.set_xlabel("Total occurrences")
ax.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

lsoa_summary

In [ ]:
# -----------------------------
# 19. Dashboard-ready output
# -----------------------------
# Priority score: simple resource allocation example.
# You can change these severity weights with your group.

severity_weights = {
    "Violence and sexual offences": 3.0,
    "Robbery": 3.0,
    "Possession of weapons": 3.0,
    "Burglary": 2.0,
    "Vehicle crime": 1.5,
    "Drugs": 1.5,
    "Theft from the person": 1.5,
    "Other theft": 1.0,
    "Shoplifting": 1.0,
    "Public order": 1.0,
    "Criminal damage and arson": 1.0,
    "Bicycle theft": 1.0,
    "Other crime": 1.0,
}

dashboard_df = results_df.copy()
dashboard_df["severity_weight"] = dashboard_df["Crime type"].map(severity_weights).fillna(1.0)
dashboard_df["priority_score"] = dashboard_df["Mixed pipeline"] * dashboard_df["severity_weight"]

dashboard_df["risk_level"] = pd.qcut(
    dashboard_df["priority_score"].rank(method="first"),
    q=3,
    labels=["Low", "Medium", "High"]
)

export_cols = [
    "Month", "LSOA code", "LSOA name", "Crime type", "Actual",
    "Simple XGBoost", "Mixed pipeline",
    "Simple XGBoost rounded", "Mixed pipeline rounded",
    "severity_weight", "priority_score", "risk_level"
]

dashboard_export = dashboard_df[export_cols].sort_values(["Month", "priority_score"], ascending=[True, False])

output_path = "forecast_comparison_dashboard_output.csv"
dashboard_export.to_csv(output_path, index=False)
print(f"Saved dashboard output to: {output_path}")

dashboard_export.head(20)

## How to interpret the result

Use the **Simple XGBoost** model as the benchmark because it represents the current teammate approach.

Use the **Mixed pipeline tuned** model as the proposed improvement because it adds:

- previous-month demand through lag features,
- short-term trends through rolling averages,
- a count-data model through Poisson regression,
- validation-based ensemble weights instead of manually guessed weights,
- and a validation-based calibration factor to reduce systematic overprediction or underprediction.

For the presentation, focus on whether the tuned mixed pipeline improves **MAE**, **RMSE**, and **WAPE**, and whether the monthly/crime-type plots follow the actual 2025 trend better than simple XGBoost.

Important: the ensemble weights are selected using **Oct-Dec 2024 validation data**, not the 2025 test data. This makes the comparison more honest.